# 📝 컨텍스트 압축 과제 LV2(응용)

교안 02의 컨텍스트 압축을 사내 규정 문서에 적용합니다. 가상 회사의 사내 규정 10개(`lv2_docs.json`, 한 규정에 9개 조항)와 질문 2개(`lv2_questions.json`)를 씁니다. 출장비·법인카드·교육비·보안 규정처럼 조항이 많고, 조건·예외·다른 규정을 가리키는 문장이 섞여 있습니다. 규정은 모두 수업용 창작입니다. 하이브리드 검색과 Qwen 리랭킹은 준비 코드로 제공하고, 원문 확인·문장 추출·답변 생성·파이프라인 연결을 직접 작성합니다.

- 1번: 원문에서 답변에 필요한 조건·예외 찾기
- 2번: 질문에 필요한 문장을 추출하고 원문과 대조
- 3번: 추출문을 근거로 답변 생성
- 4번: 새 질문을 리랭킹·추출 파이프라인으로 처리
- 5번: 파이프라인의 처리 순서와 단계별 입력·출력 설명

`.env`의 OpenAI 키가 필요합니다. 재시도·재실행을 제외하면 임베딩은 문서 10개와 질문 2회(준비 셀·4번)이고, GPT 요청은 최대 7회입니다(2번 최대 3회, 3번 최대 1회, 4번 최대 3회). 자가채점은 API를 호출하지 않습니다. 리랭커 Qwen3-Reranker-0.6B(가중치 약 1.19GB)는 로컬 CPU에서 실행하며 교안에서 받은 파일을 그대로 씁니다. 교안 02 커널은 종료하고 시작하세요.

<strong>풀이 방법</strong>: 준비 셀부터 순서대로 실행하세요. 구분선 안의 `[작성]` 부분에서 핵심 호출 코드를 작성합니다. `...`는 호출식 전체나 여러 줄의 코드로 바꿀 수 있습니다. 질문 준비·대조 표·출력 코드는 완성되어 있습니다. 문항별 핵심 동작만 자가채점하며, 추출문·답변 문장은 고정하지 않습니다. 앞 문항의 변수를 다음 문항에서 이어 씁니다.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하세요. 경로·JSON 함수·원문 ID는 지난 교안의 방식을 이어갑니다.

아래 준비 코드는 한 셀입니다. `# ====` 구분선으로 역할을 나눴습니다. 위에서부터 실행하면 규정 문서와 질문을 읽고, 교안 02와 같은 구성의 하이브리드 검색기 `hybrid`와 Qwen 리랭커 `reranker`(`top_n=3`)가 준비됩니다. 규정이 10개뿐이라 각 검색기의 후보 수는 교안의 5개 대신 3개로 줄였습니다. 마지막 구간은 질문 `question`의 후보를 한 번 검색해 리랭킹한 원문 `reranked`를 보관합니다. 1~3번은 모두 이 `reranked`를 씁니다. 이 셀을 다시 실행하면 문서를 다시 임베딩합니다.


In [ ]:
# ====================================================================
# 1) 라이브러리 가져오기
# 문서·표·토큰화·검색·답변 체인에 사용할 라이브러리를 가져옵니다.
# ====================================================================
# 본문과 출처를 같은 Document에 보관합니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# ====================================================================
# 2) 경로와 JSON 입출력
# material_dir를 기준으로 데이터를 읽고 결과를 저장하는 함수를 준비합니다.
# ====================================================================
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

# ====================================================================
# 3) 환경변수와 모델 설정
# .env의 API 키를 읽고 답변·추출에 쓸 GPT와 임베딩 모델을 설정합니다.
# ====================================================================
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 이 구간은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 아래 적재·검색 구간에서 요청합니다.")

# ====================================================================
# 4) 원문을 Document로 변환하는 함수
# 본문과 원문 ID·출처·분류를 함께 보관합니다.
# ====================================================================
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 유지하고 필터 필드만 추가합니다.
    documents = []
    for record in records:
        document = Document(
            id=record["doc_id"],
            page_content=record["text"],
            metadata={"source_id": record["doc_id"], "title": record["title"],
                      "url": record["url"], **record["metadata"]},
        )
        documents.append(document)
    return documents

# ====================================================================
# 5) 과제 데이터 읽기
# 사내 규정 10개를 읽고 Document 목록으로 바꿉니다.
# ====================================================================
# 데이터 구조는 앞 5행으로 확인하고, 검색에는 전체 문서를 사용합니다.
records = read_json("lv2_docs.json")
print("전체 문서 수:", len(records))
display(pd.DataFrame(records).head())
documents = make_documents(records)

# ====================================================================
# 6) 결과 출력과 한국어 토큰화
# show_results는 원문을 print로 보여 주고, kiwi_tokenize는 BM25에 쓸 토큰을 만듭니다.
# ====================================================================
def show_results(documents):
    """표시 순서와 원문 ID·메타데이터·본문 전체를 보여 줍니다."""
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    print("결과 문서 수:", len(documents))
    for order, doc in enumerate(documents, start=1):
        print(f"[{order}] 원문 ID:", doc.metadata["source_id"])
        print("메타데이터:", doc.metadata)
        # 본문은 별도 줄에 출력해 원문의 줄바꿈을 그대로 읽습니다.
        print("본문:")
        print(doc.page_content)
        print()

# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF에서 온 반각 가운뎃점(･)은 분석기가 앞뒤를 다른 낱말로 끊으므로 가운뎃점(·)으로 바꿉니다.
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    words = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith("N") or token.tag in {"SL", "SN"}:
            words.append(token.form.lower())
    return words

# ====================================================================
# 7) 질문 읽기
# 사람이 원문을 읽고 정한 질문과 근거 원문 ID를 읽습니다.
# ====================================================================
# 사람이 원문을 읽고 정한 평가 질문과 근거 원문 ID(evidence)입니다.
questions = read_json("lv2_questions.json")
question_by_id = {row["question_id"]: row for row in questions}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(questions)[["question_id", "question", "evidence"]])

# ====================================================================
# 8) 하이브리드 검색기
# 문서를 임베딩해 Chroma에 적재하고 BM25·Dense 검색기를 RRF로 연결합니다.
# ====================================================================
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day48_lv2_rules", embedding_function=embedding_model)
vector_store.reset_collection()
# 같은 원문 ID를 저장소 ID와 검색 결과 메타데이터에 함께 유지합니다.
added_ids = vector_store.add_documents(documents, ids=[doc.metadata["source_id"] for doc in documents])
print("day48_lv2_rules 적재 수:", len(added_ids))

# 규정 10개에서 각 검색기가 최대 3개씩 찾으므로 합친 후보는 3~6개입니다. 리랭커가 그중 3개를 고릅니다.
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=3)
dense = vector_store.as_retriever(search_kwargs={"k": 3})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")

# ====================================================================
# 9) 리랭킹·추출 구성요소와 모델
# 공식 리랭커·추출기 인터페이스를 가져오고 교안과 같은 Cross-Encoder 리랭커를 준비합니다.
# ====================================================================
# 공식 리랭커 인터페이스를 사용합니다.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# 문서 선택 뒤 질문 관련 문장만 추출하는 공식 구성요소입니다.
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, DocumentCompressorPipeline

# 첫 실행에는 공개 모델 파일을 내려받습니다. 노트북마다 한 번만 준비합니다.
# 실습은 CPU와 입력 상한 1024토큰을 사용합니다. 긴 입력은 잘릴 수 있습니다.
cross_encoder = HuggingFaceCrossEncoder(
    model_name="Qwen/Qwen3-Reranker-0.6B",
    model_kwargs={"device": "cpu", "max_length": 1024},
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# ====================================================================
# 10) 추출에 사용할 원문 선택
# 질문의 후보를 한 번 검색하고 리랭킹한 원문을 reranked에 보관합니다.
# ====================================================================
# 1~3번에서 함께 쓸 질문입니다(lv2_questions.json의 r01).
question = "해외 출장 중 법인카드 결제가 거절되어 호텔비를 개인 카드로 냈어요. 언제까지 무엇을 내면 정산받을 수 있나요?"

# 후보를 검색하고 리랭킹한 원문을 보관한 뒤, 질문에 필요한 문장을 추출합니다.
candidates = hybrid.invoke(question)
reranked = list(reranker.compress_documents(candidates, question))
print("후보 ID:", [doc.metadata["source_id"] for doc in candidates])
print("선택 원문 ID:", [doc.metadata["source_id"] for doc in reranked])


## 1. 원문에서 답변에 필요한 조건·예외를 찾습니다

<strong>배경</strong>: 질문에 답하는 데 필요한 조건·예외를 원문에서 먼저 찾습니다. 조건·예외·다른 규정을 가리키는 문장이 빠지면 답이 달라집니다.

<strong>요구사항</strong>:

- <strong>`review_criteria`</strong>: 확인 기준 딕셔너리 3개를 담은 목록입니다. 키는 `item`(항목 이름 문자열), `source_id`(원문 ID), `fact`(원문에서 그대로 복사한 구절) 세 개입니다.
  - `fact`는 한 조항 안에서 고른 10자 이상의 구절이며 철자·띄어쓰기까지 원문과 같게 복사합니다. 규정 본문은 조항마다 줄이 바뀌므로 두 조항에 걸쳐 복사하면 원문과 달라집니다.
  - 기준은 규정 원문 전체에서 고릅니다. 그 원문이 선택 문서(`reranked`)에 들어 있는지는 제공 코드가 보여 줍니다.
  - 세 기준 중 하나 이상은 `않는다`나 `다만`이 든 예외·부정 구절로 고릅니다.

<strong>확인 기준</strong>: `fact`가 해당 `source_id`의 원문에 그대로 있으면 통과합니다. 어떤 구절을 고를지는 판단이며, 2번에서 추출문에 남았는지 확인합니다. 제공 코드가 각 기준의 원문이 선택 문서(`reranked`)에 들어 있는지도 보여 줍니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문의 답을 바꾸는 조항을 원문에서 찾아 그대로 복사합니다.

세부구현:
1. 출력된 reranked 원문에서 질문의 '언제까지'와 '무엇을'에 답하는 조항을 찾습니다.
2. 그 답을 바꾸는 예외, 금액 조건, 다른 규정을 가리키는 조항도 찾습니다.
3. 조항의 구절을 복사해 fact에, 그 원문 ID를 source_id에, 무엇을 확인하는 기준인지 item에 적습니다.
```

</details>


In [ ]:
# 1) 선택된 원문을 끝까지 읽고 질문의 답에 필요한 조항을 찾습니다.
show_results(reranked)

# ====================================================================
# [작성] 2) 확인 기준 3개를 채우세요. fact는 원문에서 그대로 복사합니다.
# 모양: {"item": "항목 이름", "source_id": "원문 ID", "fact": "원문에서 복사한 구절"}
review_criteria = [
    {"item": ..., "source_id": ..., "fact": ...},
    {"item": ..., "source_id": ..., "fact": ...},
    {"item": ..., "source_id": ..., "fact": ...},
]
# ====================================================================

# 3) 각 기준의 원문이 선택 문서에 들어 있는지 확인합니다. 없으면 검색·선택 단계에서 빠진 것입니다.
selected_ids = [doc.metadata["source_id"] for doc in reranked]
display(pd.DataFrame([{**row, "selected": row["source_id"] in selected_ids} for row in review_criteria]))


In [ ]:
# [자가채점]
doc_text = {doc.metadata["source_id"]: doc.page_content for doc in documents}
assert isinstance(review_criteria, list) and len(review_criteria) == 3, "확인 기준 딕셔너리 3개를 review_criteria 목록에 담으세요."
assert all(isinstance(row, dict) and set(row) == {"item", "source_id", "fact"} for row in review_criteria), "각 기준은 item, source_id, fact 세 키를 가진 딕셔너리입니다."
assert all(isinstance(row["item"], str) and row["item"].strip() for row in review_criteria), "item에 확인할 항목 이름을 문자열로 적으세요."
assert all(row["source_id"] in doc_text for row in review_criteria), "source_id는 lv2_docs.json에 있는 원문 ID(예: reg02)여야 합니다."
assert all(isinstance(row["fact"], str) and len(row["fact"]) >= 10 for row in review_criteria), "fact에는 10자 이상의 구절을 문자열로 담으세요."
assert all(row["fact"] in doc_text[row["source_id"]] for row in review_criteria), "fact가 해당 원문에 그대로 없습니다. 한 조항 안에서 철자·띄어쓰기까지 그대로 복사했는지, source_id가 그 원문의 ID인지 확인하세요."
assert any("않는다" in row["fact"] or "다만" in row["fact"] for row in review_criteria), "세 기준 중 하나 이상은 '않는다'나 '다만'이 든 예외·부정 구절로 고르세요."
print("✅ 1번 핵심 확인 완료!")


아래는 교안 02의 원문·추출문 대조 함수 `excerpt_rows`입니다. 입력 원문을 모두 남기고 같은 ID의 추출문, 유지 여부(`retained`), 추출문이 원문의 연속 부분문자열인지(`contiguous_match`)를 한 행에 담습니다.


In [ ]:
def excerpt_rows(before, after):
    """선택한 원문 전체를 남기고 발췌 유지·누락 여부를 나란히 보여 줍니다."""
    excerpts = {}
    for doc in after:
        source_id = doc.metadata["source_id"]
        excerpts[source_id] = doc.page_content
    rows = []
    for doc in before:
        source_id = doc.metadata["source_id"]
        retained = source_id in excerpts
        excerpt = excerpts.get(source_id, "")
        rows.append({"source_id": source_id, "original": doc.page_content,
                     "excerpt": excerpt, "retained": retained,
                     "contiguous_match": excerpt in doc.page_content if retained else None})
    # 누락된 문서도 표에 남깁니다. 일치 여부만으로 의미 보존을 판정하지 않습니다.
    return rows


## 2. 질문에 필요한 문장을 추출하고 원문과 대조합니다

<strong>배경</strong>: 선택한 원문에서 질문에 필요한 문장만 추출한 뒤, 1번에서 찾은 조건·예외가 추출문에 남았는지 확인합니다.

<strong>요구사항</strong>:

- <strong>`extractor`</strong>: `LLMChainExtractor.from_llm`에 준비된 `llm`을 넣어 만든 추출기입니다.
- <strong>`compressed`</strong>: `extractor.compress_documents`에 `reranked`와 `question`을 넣은 결과를 `list`로 바꾼 목록입니다.
- <strong>`excerpt_table`</strong>: `excerpt_rows`에 `reranked`를 before, `compressed`를 after로 넣은 결과 목록입니다.

<strong>확인 기준</strong>: `compressed`는 0~3개의 `Document`이며 원문 ID를 유지합니다. 추출에서 통째로 빠진 원문도 `excerpt_table`에 `retained=False`로 남습니다. 1번 기준의 구절이 추출문에 없으면, 다른 표현으로 남았는지 내용이 빠졌는지 직접 읽고 확인합니다. 추출 결과 자체를 고정된 정답으로 채점하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 추출기를 만든 뒤 선택 원문과 질문을 전달하고, 같은 원문 ID의 추출 결과를 대조합니다.

세부구현:
1. LLMChainExtractor.from_llm에 llm을 전달합니다.
2. compress_documents에 문서 목록과 질문을 순서대로 넣고 결과를 list로 바꿉니다.
3. excerpt_rows에 원문 목록과 추출 목록을 순서대로 넣습니다.
```

</details>


In [ ]:
# ====================================================================
# [작성] 1) LLMChainExtractor.from_llm(llm)으로 extractor를 만드세요.
# [작성] 2) extractor.compress_documents에 reranked와 question을 넣으세요.
#          결과를 list로 바꿔 compressed에 담고, 원문 reranked는 덮어쓰지 않습니다.
#          첫 인자는 문서 목록, 두 번째 인자는 질문 문자열입니다.
extractor = ...
compressed = ...
# ====================================================================

# ====================================================================
# [작성] 3) excerpt_rows에 reranked를 before, compressed를 after로 넣어 excerpt_table에 담으세요.
excerpt_table = ...
# ====================================================================

# 4) 원문과 추출문을 끝까지 나란히 읽습니다. retained=False는 문서 전체가 빠졌다는 뜻입니다.
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(excerpt_table))

# 5) 1번 기준이 추출문에 글자 그대로 남았는지 봅니다. 표현이 달라도 뜻은 남았을 수 있으니 직접 읽습니다.
excerpt_by_id = {row["source_id"]: row["excerpt"] for row in excerpt_table}
criteria_kept = [
    {"item": row["item"], "source_id": row["source_id"],
     "kept_in_excerpt": row["fact"] in excerpt_by_id.get(row["source_id"], "")}
    for row in review_criteria
]
display(pd.DataFrame(criteria_kept))


In [ ]:
# [자가채점]
assert isinstance(extractor, LLMChainExtractor), "LLMChainExtractor.from_llm(llm)으로 extractor를 만드세요."
assert all(any(doc.page_content == original.page_content and doc.metadata == original.metadata for original in candidates) for doc in reranked), "reranked가 원문이 아닙니다. 추출 결과로 덮어쓰지 말고 준비 셀부터 다시 실행하세요."
assert type(compressed) is list and all(isinstance(doc, Document) for doc in compressed), "compress_documents의 결과를 list()로 감싸 compressed에 담으세요."
assert len(compressed) <= len(reranked), "reranked를 추출기에 전달한 결과만 compressed에 담으세요."
assert {doc.metadata["source_id"] for doc in compressed} <= {doc.metadata["source_id"] for doc in reranked}, "추출 입력은 reranked입니다. 다른 후보나 새 검색 결과를 넣지 마세요."
assert excerpt_table == excerpt_rows(reranked, compressed), "excerpt_rows에는 reranked를 before, compressed를 after로 넣으세요."
print("✅ 2번 핵심 확인 완료!")


아래는 교안 02의 답변 문맥 함수 `format_context`와 답변 체인 `answer_chain`입니다. 추출문에 원문 ID를 붙여 답변의 근거로 전달합니다.


In [ ]:
# 출처와 본문을 함께 전달해 답변을 실제 원문으로 되짚을 수 있게 합니다.
def format_context(documents):
    """실제로 검색한 원문만 ID·제목·출처와 함께 답변 문맥으로 만듭니다."""
    context_parts = []
    for doc in documents:
        context = (
            f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n"
            f"출처: {doc.metadata['url']}\n{doc.page_content}"
        )
        context_parts.append(context)
    return "\n\n".join(context_parts)


In [ ]:
# 전달받은 문맥을 바탕으로 답변하며 근거의 원문 ID를 인용합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 원문 근거로만 답하세요. 주장마다 [원문 ID]를 인용하세요. "
               "근거에 답이 없으면 확인할 수 없다고 말하세요. 자료의 빈 부분을 추측하지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()


## 3. 추출문을 근거로 답변을 생성합니다

<strong>배경</strong>: 추출한 문장과 원문 ID를 답변 체인에 전달해, 근거를 인용하는 답변을 만듭니다.

<strong>요구사항</strong>:

- <strong>`compressed_answer`</strong>: `answer_chain.invoke`에 `question`과 `format_context(compressed)`를 `question`·`context` 키로 넣어 받은 답변 문자열입니다. 추출 근거가 없을 때의 분기는 제공 코드입니다.

<strong>확인 기준</strong>: 추출 근거가 있으면 답변과 인용 원문 ID를 읽고 1번에서 찾은 조건·예외와 맞는지 확인합니다. 추출 결과가 비었으면 답변은 `"전달할 근거가 없어 답변을 생성하지 않았습니다."`이고 모델을 부르지 않습니다. 두 경우 모두 정상입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 추출문 목록을 답변 문맥으로 바꿔 질문과 함께 전달합니다.

세부구현:
1. format_context로 compressed의 본문과 원문 ID를 문맥 문자열로 만듭니다.
2. question과 context 키를 가진 딕셔너리를 answer_chain.invoke에 전달합니다.
```

</details>


In [ ]:
# 1) 추출 근거가 없으면 모델에 답변을 요청하지 않습니다.
if compressed:
    # ====================================================================
    # [작성] 2) answer_chain.invoke에 question과 format_context(compressed)를 question·context 키로 넣으세요.
    #          반환된 답변 문자열을 compressed_answer에 담습니다.
    compressed_answer = ...
    # ====================================================================
else:
    compressed_answer = "전달할 근거가 없어 답변을 생성하지 않았습니다."

# 3) 답변의 조건·예외와 인용 ID를 앞에서 확인한 원문·추출문과 대조합니다.
print(compressed_answer)


In [ ]:
# [자가채점]
assert isinstance(compressed_answer, str) and compressed_answer.strip(), "answer_chain.invoke의 반환값을 compressed_answer에 담으세요."
assert compressed or compressed_answer == "전달할 근거가 없어 답변을 생성하지 않았습니다.", "추출 근거가 없을 때의 제공 분기를 지우지 말고 그대로 두세요."
print("✅ 3번 핵심 확인 완료!")


## 4. 새 질문을 리랭킹·추출 파이프라인으로 처리합니다

<strong>배경</strong>: 새 질문마다 검색·리랭킹·추출을 따로 부르지 않도록 후처리 순서를 하나로 묶어 검색기에 연결합니다. `transformers` 목록의 순서가 실행 순서입니다.

<strong>요구사항</strong>:

- <strong>`compressor`</strong>: `transformers`에 `reranker`, 2번 `extractor`를 이 순서로 넣은 `DocumentCompressorPipeline`입니다.
- <strong>`compression_retriever`</strong>: `base_retriever`에 `hybrid`, `base_compressor`에 `compressor`를 넣은 `ContextualCompressionRetriever`입니다.
- <strong>`next_compressed`</strong>: `compression_retriever`로 준비된 `next_question`을 한 번 `invoke`한 결과입니다. 1~3번의 변수는 덮어쓰지 않습니다.

<strong>확인 기준</strong>: `next_compressed`는 0~3개의 추출 문서입니다. 이 호출 하나가 검색·리랭킹·추출을 모두 실행하며 추출 요청이 최대 3회 나갑니다.

<details><summary>힌트</summary>

```text
접근방법:
- 후처리 객체를 순서대로 묶은 뒤, 그 묶음을 검색기의 후처리 자리에 연결합니다.

세부구현:
1. DocumentCompressorPipeline의 transformers에 두 객체를 목록으로 넣습니다.
2. ContextualCompressionRetriever에 hybrid와 compressor를 연결합니다.
3. 만든 검색기의 invoke에 next_question을 전달합니다.
```

</details>


In [ ]:
# 1) 1~3번과 다른 새 질문입니다(lv2_questions.json의 r02).
next_question = "회사 지원으로 외부 교육을 듣고, 교육이 끝난 지 6개월 만에 이직하면 지원금을 돌려줘야 하나요?"

# ====================================================================
# [작성] 2) DocumentCompressorPipeline(transformers=[...])에 reranker, extractor 순서로 넣어 compressor를 만드세요.
# [작성] 3) ContextualCompressionRetriever에 base_retriever=hybrid, base_compressor=compressor를 지정하세요.
# [작성] 4) compression_retriever.invoke(next_question)의 결과를 next_compressed에 담으세요.
compressor = ...
compression_retriever = ...
next_compressed = ...
# ====================================================================

# 5) 어느 원문에서 무엇이 추출됐는지 확인합니다. 사람이 정한 근거 원문 ID와 비교합니다.
print("근거 원문 ID:", question_by_id["r02"]["evidence"])
show_results(next_compressed)
# 6) 1번처럼 기준을 먼저 정해 두고 추출문에 남았는지 봅니다. 이 예외가 빠지면 반환 여부를 잘못 안내합니다.
key_exception = "다만 권고사직이나 회사 사정에 따른 퇴사는 반환하지 않으며"
print("예외 조항이 추출문에 남았는지:", any(key_exception in doc.page_content for doc in next_compressed))


In [ ]:
# [자가채점]
assert isinstance(compressor, DocumentCompressorPipeline) and len(compressor.transformers) == 2, "DocumentCompressorPipeline의 transformers에 reranker와 extractor 두 객체를 모두 넣으세요."
assert compressor.transformers == [reranker, extractor], "transformers에는 reranker 다음 extractor 순서로 넣으세요. 순서가 곧 실행 순서입니다."
assert isinstance(compression_retriever, ContextualCompressionRetriever) and compression_retriever.base_retriever is hybrid and compression_retriever.base_compressor is compressor, "base_retriever에는 hybrid, base_compressor에는 compressor를 연결하세요."
assert isinstance(next_compressed, list) and len(next_compressed) <= 3 and all(isinstance(doc, Document) for doc in next_compressed), "compression_retriever.invoke(next_question)의 결과(최대 3개)를 next_compressed에 담으세요."
assert {doc.metadata["source_id"] for doc in next_compressed} <= {doc.metadata["source_id"] for doc in documents}, "next_compressed는 제공된 원문 ID를 유지해야 합니다."
assert next_compressed is not compressed, "next_compressed가 2번의 compressed와 같은 객체입니다. compression_retriever.invoke(next_question)로 새로 받으세요."
assert {doc.metadata["source_id"] for doc in compressed} <= {doc.metadata["source_id"] for doc in reranked}, "2번의 compressed가 바뀌었습니다. 4번 결과는 next_compressed에만 받고 2번부터 다시 실행하세요."
print("✅ 4번 핵심 확인 완료!")


## 5. 파이프라인의 처리 순서와 단계별 입력·출력을 설명합니다

<strong>배경</strong>: 4번에서 연결한 검색기가 질문을 받아 추출 문서를 돌려주기까지 어떤 자료가 각 단계에 전달되는지 정리합니다.

<strong>요구사항</strong>:

- <strong>설명</strong>: `hybrid`, `reranker`, `extractor` 순서로 각 단계의 입력과 출력을 한 문장씩 쓰세요. 검색 후보, 선택한 원문, 추출문이 어떻게 이어지는지 설명합니다.

<strong>확인 기준</strong>: 세 단계에 대해 총 세 문장으로 씁니다. 원문 ID는 추출 뒤에도 유지된다는 점을 포함합니다. 코드는 작성하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4번의 base_retriever와 transformers 목록을 따라 입력과 출력을 확인합니다.

세부구현:
1. hybrid가 질문을 받아 무엇을 검색하는지 확인합니다.
2. reranker가 후보에서 무엇을 고르는지 확인합니다.
3. extractor가 선택 문서에서 무엇을 남기며 어떤 메타데이터를 유지하는지 확인합니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*
